# EloKriti candidate-pool construction

This notebook consolidates the candidate-library construction steps used for the
EloKriti study:

**OMG dataset → mechanism/MW filtering → bisphenol-family exclusion →
canonical-structure deduplication → GAFF-compatible set → Morgan/Butina
diversity selection → final 1000-candidate pool.**


## Expected inputs

1. `OMG_ML_pred_std_with_normalization_publish.csv`  
   Original OMG-derived table used for mechanism and molecular-weight filtering.

2. `gaff_ok.csv`  
   GAFF-compatible subset produced after canonical deduplication and
   parameterization screening. The corresponding study-specific file should be
   deposited with the research-data archive.

The GAFF-compatibility screening itself was performed outside the two source
notebooks supplied for this consolidation. Therefore, this notebook does **not**
invent a replacement implementation for that step; instead it validates the
deposited `gaff_ok.csv` and continues the diversity-selection workflow
from that recorded screening result.

## Expected funnel from the original run

- OMG rows: **12,886,131**
- mechanism 8: **29,565**
- mechanism 8 + MW 100–600 g/mol: **29,540**
- after BPA/BPF/BPS/BPAF structural exclusions: **29,379**
- after canonical-structure deduplication: **29,337**
- GAFF-compatible candidates: **28,632**
- Butina clusters at cutoff 0.70: **824**
- final diversity-selected pool: **1000**


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from rdkit.ML.Cluster import Butina

OMG_CSV = Path("OMG_ML_pred_std_with_normalization_publish.csv")
GAFF_OK_CSV = Path("gaff_ok.csv")

MECHANISM = 8
MW_MIN = 100.0
MW_MAX = 600.0

RADIUS = 2
NBITS = 2048
BUTINA_CUTOFF = 0.70
TARGET_POOL_SIZE = 1000


## 1. Filter the OMG-derived table by polymerization mechanism and molecular weight

The reported screening library used `polymerization_mechanism_idx == 8` and
molecular weight between 100 and 600 g/mol, inclusive.


In [ ]:
usecols = [
    "methyl_terminated_product",
    "reaction_id",
    "product",
    "reactant_1",
    "reactant_2",
    "polymerization_mechanism_idx",
    "molecular_weight_Boltzmann_average",
    "logP_Boltzmann_average",
    "TPSA_Boltzmann_average",
    "s1_energy_Boltzmann_average",
    "chi_parameter_water_mean",
    "normalized_monomer_phi_Boltzmann_average",
    "normalized_backbone_phi_Boltzmann_average",
]

mechanism_count = 0
total_rows = 0
kept_chunks = []

for chunk in pd.read_csv(OMG_CSV, usecols=usecols, chunksize=500_000):
    total_rows += len(chunk)

    mech_mask = chunk["polymerization_mechanism_idx"].eq(MECHANISM)
    mechanism_count += int(mech_mask.sum())

    mask = mech_mask.copy()
    mask &= chunk["molecular_weight_Boltzmann_average"].between(
        MW_MIN, MW_MAX, inclusive="both"
    )

    kept_chunks.append(chunk.loc[mask].copy())

filtered = pd.concat(kept_chunks, ignore_index=True)

print(f"Total rows scanned: {total_rows:,}")
print(f"Mechanism {MECHANISM}: {mechanism_count:,}")
print(f"Mechanism {MECHANISM} + MW {MW_MIN:g}-{MW_MAX:g}: {len(filtered):,}")

assert total_rows == 12_886_131
assert mechanism_count == 29_565
assert len(filtered) == 29_540


## 2. Exclude BPA/BPF/BPS/BPAF bridge motifs

The four SMARTS patterns below are the structural blacklist used to remove
bisphenol-family bridge motifs from the filtered screening space. Matching is
performed on `methyl_terminated_product`, which is the parseable
methyl-terminated representation used in the original filtering notebook.


In [ ]:
SMARTS_BLACKLIST = {
    "BPA":  "c1ccc(cc1)C(C)(C)c2ccc(cc2)",
    "BPF":  "c1ccc(cc1)C(c2ccc(cc2))",
    "BPS":  "c1ccc(cc1)S(=O)(=O)c2ccc(cc2)",
    "BPAF": "c1ccc(cc1)C(C(F)(F)F)(C(F)(F)F)c2ccc(cc2)",
}

patterns = {name: Chem.MolFromSmarts(smarts)
            for name, smarts in SMARTS_BLACKLIST.items()}

if any(p is None for p in patterns.values()):
    raise ValueError("One or more SMARTS patterns could not be parsed.")

mols = [Chem.MolFromSmiles(str(s))
        for s in filtered["methyl_terminated_product"]]

valid_mask = np.array([m is not None for m in mols])
if not valid_mask.all():
    print("Unparseable methyl-terminated structures:",
          int((~valid_mask).sum()))

df_valid = filtered.loc[valid_mask].copy().reset_index(drop=True)
valid_mols = [m for m in mols if m is not None]

keep = []
for mol in valid_mols:
    matched = any(mol.HasSubstructMatch(patt) for patt in patterns.values())
    keep.append(not matched)

bisphenol_filtered = df_valid.loc[keep].copy().reset_index(drop=True)

print(f"Input to structural exclusion: {len(df_valid):,}")
print(f"Removed by BPA/BPF/BPS/BPAF blacklist: "
      f"{len(df_valid) - len(bisphenol_filtered):,}")
print(f"Remaining: {len(bisphenol_filtered):,}")

assert len(bisphenol_filtered) == 29_379


## 3. Canonicalize and deduplicate structures

Canonicalization is performed on `methyl_terminated_product`. Rows that map to
the same canonical chemical structure are collapsed by retaining the first
occurrence, matching the original molecule-centric curation step.


In [ ]:
def canonicalize_smiles(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    return Chem.MolToSmiles(
        mol,
        isomericSmiles=True,
        canonical=True,
    )

dedup = bisphenol_filtered.copy()
dedup["mt_canonical_smiles"] = (
    dedup["methyl_terminated_product"].map(canonicalize_smiles)
)

dedup = dedup[dedup["mt_canonical_smiles"].notna()].copy()
dedup = dedup.drop_duplicates(
    subset=["mt_canonical_smiles"],
    keep="first",
).reset_index(drop=True)

print(f"Canonical unique structures: {len(dedup):,}")

assert len(dedup) == 29_337

dedup.to_csv("filtered_unique_29337.csv", index=False)


## 4. GAFF-compatibility handoff

The original diversity-selection notebook begins from `gaff_ok.csv`, containing
the structures that passed the study's GAFF parameterization screening.

Because the two source notebooks do not contain the code that generated the
GAFF pass/fail table, this consolidation keeps that stage explicit rather than
reconstructing it from assumptions. The deposited `gaff_ok.csv` is treated as
the recorded output of that screening step.

The checks below ensure that the file has the expected size, mechanism, and
compatibility flag before diversity selection.


In [ ]:
gaff_df = pd.read_csv(GAFF_OK_CSV)

print(f"GAFF-compatible rows: {len(gaff_df):,}")

assert len(gaff_df) == 28_632
assert set(gaff_df["polymerization_mechanism_idx"].dropna().unique()) == {8}

if "GAFF_compatible" in gaff_df.columns:
    assert gaff_df["GAFF_compatible"].fillna(False).astype(bool).all()

# Optional provenance check against the preceding canonical-deduplicated set.
if "reaction_id" in gaff_df.columns and "reaction_id" in dedup.columns:
    missing_ids = set(gaff_df["reaction_id"]) - set(dedup["reaction_id"])
    print("GAFF-compatible reaction IDs absent from reconstructed "
          f"29,337-member set: {len(missing_ids)}")
    assert len(missing_ids) == 0


## 5. Compute Morgan fingerprints

The diversity-selection stage uses Morgan fingerprints with radius 2 and 2048
bits, computed from `reactant_1`, matching the original selection notebook.


In [ ]:
SMILES_COL = "reactant_1"
ID_COL = "reaction_id"

work = gaff_df.dropna(subset=[SMILES_COL]).copy().reset_index(drop=True)

mols = []
valid_idx = []

for i, smi in enumerate(work[SMILES_COL].astype(str)):
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        mols.append(mol)
        valid_idx.append(i)

work = work.iloc[valid_idx].reset_index(drop=True)

print(f"Valid molecules for fingerprinting: {len(work):,}")
assert len(work) == 28_632

fps = [
    AllChem.GetMorganFingerprintAsBitVect(mol, RADIUS, nBits=NBITS)
    for mol in tqdm(mols, desc="Morgan fingerprints")
]

assert len(fps) == 28_632


## 6. Butina clustering

Pairwise Tanimoto similarities are converted to distances as
`1 - similarity`. The original workflow tested Butina distance cutoffs of
0.60, 0.65, 0.70, and 0.75 and selected **0.70**, which yielded 824 clusters.


In [ ]:
def tanimoto_distance_matrix(fp_list):
    distances = []
    for i in tqdm(range(1, len(fp_list)), desc="Tanimoto distances"):
        similarities = DataStructs.BulkTanimotoSimilarity(
            fp_list[i], fp_list[:i]
        )
        distances.extend(1.0 - s for s in similarities)
    return distances

dist_matrix = tanimoto_distance_matrix(fps)

expected_cluster_counts = {
    0.60: 2263,
    0.65: 1429,
    0.70: 824,
    0.75: 402,
}

for cutoff, expected in expected_cluster_counts.items():
    c = Butina.ClusterData(
        dist_matrix,
        len(fps),
        cutoff,
        isDistData=True,
    )
    print(f"{cutoff:.2f} -> {len(c):,} clusters")
    assert len(c) == expected


## 7. Build the 1000-candidate diversity pool

At the selected cutoff of 0.70:

1. one Butina representative (`cluster[0]`) is taken from each of the 824
   clusters;
2. the remaining 176 positions are filled by greedy MaxMin/farthest-point
   selection, maximizing each candidate's minimum Tanimoto distance to the
   already selected set.

No objective values from the MD campaign enter this 1000-member diversity
selection.


In [ ]:
clusters = Butina.ClusterData(
    dist_matrix,
    len(fps),
    BUTINA_CUTOFF,
    isDistData=True,
)
clusters = sorted(clusters, key=len, reverse=True)

selected_indices = [cluster[0] for cluster in clusters]

print("Butina representatives:", len(selected_indices))
assert len(selected_indices) == 824

remaining_needed = TARGET_POOL_SIZE - len(selected_indices)
all_indices = set(range(len(fps)))
selected_set = set(selected_indices)
remaining_pool = sorted(all_indices - selected_set)

def min_distance_to_selected(idx, selected, fp_list):
    similarities = DataStructs.BulkTanimotoSimilarity(
        fp_list[idx],
        [fp_list[j] for j in selected],
    )
    return 1.0 - max(similarities)

for _ in tqdm(range(remaining_needed), desc="Greedy MaxMin fill"):
    best_idx = None
    best_dist = -1.0

    for idx in remaining_pool:
        distance = min_distance_to_selected(
            idx,
            selected_indices,
            fps,
        )
        if distance > best_dist:
            best_dist = distance
            best_idx = idx

    selected_indices.append(best_idx)
    selected_set.add(best_idx)
    remaining_pool.remove(best_idx)

assert len(selected_indices) == TARGET_POOL_SIZE
print("Final selected candidates:", len(selected_indices))


## 8. Save the final candidate pool and cluster summary


In [ ]:
selected_df = work.iloc[selected_indices].copy().reset_index(drop=True)
selected_df.to_csv("selected_1000.csv", index=False)

candidate_cols = [
    "reaction_id",
    "reactant_1",
    "molecular_weight_Boltzmann_average",
    "logP_Boltzmann_average",
    "TPSA_Boltzmann_average",
    "normalized_monomer_phi_Boltzmann_average",
    "normalized_backbone_phi_Boltzmann_average",
    "polymerization_mechanism_idx",
    "GAFF_compatible",
]

available_cols = [c for c in candidate_cols if c in selected_df.columns]
candidate_pool = selected_df[available_cols].copy()

if "GAFF_compatible" in candidate_pool.columns:
    candidate_pool = candidate_pool[
        candidate_pool["GAFF_compatible"].fillna(False).astype(bool)
    ].reset_index(drop=True)

assert len(candidate_pool) == 1000
candidate_pool.to_csv("candidate_pool.csv", index=False)

cluster_summary = pd.DataFrame({
    "cluster_id": range(len(clusters)),
    "cluster_size": [len(c) for c in clusters],
    "representative_index": [c[0] for c in clusters],
    ID_COL: [work.iloc[c[0]][ID_COL] for c in clusters],
    SMILES_COL: [work.iloc[c[0]][SMILES_COL] for c in clusters],
})
cluster_summary.to_csv("cluster_summary_cutoff_0.7.csv", index=False)

print("Saved:")
print("  selected_1000.csv")
print("  candidate_pool.csv")
print("  cluster_summary_cutoff_0.7.csv")
